# 0.06 Test site: SRTM DEM download and preprocessing

**Purpose.** Test how achievable, how slow and how large it is to obtain the SRTM elevation and slope input for one
Geo-Wiki site (a 300 x 300 m area in Africa, i.e. the 100 x 100 m Geo-Wiki site plus a 100 m margin), so the cost for all
sites can be extrapolated. Third per-modality notebook after [0.04 Sentinel-1](0.04-tp-test-site-sentinel1.ipynb) and
[0.05 Sentinel-2](0.05-tp-test-site-sentinel2.ipynb).

**Inputs.** Geo-Wiki site centre from `data/interim/geowiki/14871660/site_summary.csv` (notebook 0.03);
Google Earth Engine image `USGS/SRTMGL1_003` (30 m, collected February 2000); project-wide settings in
`tomer_peled_mgi_thesis.config`.

**Outputs.** Raw GeoTIFF in `data/raw/sites/{site_id}/srtm/`, normalised Galileo-style array in
`data/processed/sites/{site_id}/`, timing/storage tables in `runs/tables/test_site/`, log in `runs/`.

**Method source.** The Earth Engine and preprocessing code is adapted from the Galileo repository
([nasaharvest/galileo](https://github.com/nasaharvest/galileo), MIT licence, commit `0f0b5b9`):
`src/data/earthengine/srtm.py` (`get_single_srtm_image`, bands and normalisation constants),
`src/data/earthengine/eo.py` (`url`-mode download), `src/data/earthengine/ee_bbox.py` (`EEBoundingBox`) and
`src/data/dataset.py` (`_tif_to_array` space-band branch, `_fillna`, `Normalizer`, `save_h5py`). Deviations are marked
`# changed:`. As in the thesis proposal the DEM is static in time: elevation and slope form one 2-band raster, resampled to
the 10 m grid by nearest neighbour, so the study window and 30-day steps do not apply.

```mermaid
flowchart LR
    A["site_summary.csv: site centre"] --> B["300 x 300 m bounding box (site + margin)"]
    B --> C["GEE USGS/SRTMGL1_003: elevation, slope from elevation"]
    C --> D["getDownloadURL: GeoTIFF at 10 m, 2 bands"]
    D --> E["data/raw/sites/{site_id}/srtm/ .tif"]
    E --> F["fill NaN, reshape h w c, normalise, float16"]
    F --> G["data/processed/sites/{site_id}/ .h5"]
    D -.timed.-> H["runs/tables/test_site/ timings and storage"]
    F -.timed.-> H
```

## 1. Settings

Project-wide values come from the package config (site id, site size, Earth Engine login variables). Only
notebook-specific choices are set here.

| Parameter | Default | Unit | Meaning |
| --- | --- | --- | --- |
| `TEST_SITE_ID` | 1151194 | Geo-Wiki sample id | Site under test (config) |
| `SITE_SIZE_METRES` | 300 | m | Exported area edge length (config) |
| `N_DOWNLOAD_REPEATS` | 3 | runs | Repeated download requests, to see timing variability |
| `SRTM_DIV_VALUES` | 2000, 50 | m, degrees | Galileo's fixed normalisation (about the 90th percentile of elevation and slope) |

In [ ]:
import time, shutil, warnings
from math import cos, radians

from googleapiclient.errors import HttpError
import ee, h5py, numpy as np, pandas as pd, rasterio, requests, xarray as xr
import rioxarray  # registers the rasterio engine
from einops import rearrange, repeat
from IPython.display import display

from tomer_peled_mgi_thesis import config as cfg
from tomer_peled_mgi_thesis.run_logging import log_run

SITE_DIR_RAW = cfg.RAW_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID) / "srtm"
SITE_DIR_PROCESSED = cfg.PROCESSED_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID)
TABLES_DIR = cfg.TABLES_DIR / "test_site"
N_DOWNLOAD_REPEATS = 3

# Galileo srtm.py settings
SRTM_IMAGE = "USGS/SRTMGL1_003"
SRTM_BANDS = ["elevation", "slope"]
SRTM_SHIFT_VALUES = np.array([0.0, 0.0])
SRTM_DIV_VALUES = np.array([2000.0, 50.0])

for d in (SITE_DIR_RAW, SITE_DIR_PROCESSED, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
log_ctx = log_run(cfg.RUNS_DIR / f"{stamp}-test-site-srtm.log")
logger = log_ctx.__enter__()  # closed in the summary cell
timings = []  # rows: step, phase, seconds

class timed:
    def __init__(self, step, phase): self.step, self.phase = step, phase
    def __enter__(self): self.t = time.perf_counter()
    def __exit__(self, *exc):
        secs = time.perf_counter() - self.t
        timings.append({"step": self.step, "phase": self.phase, "seconds": secs})
        logger.info("%s [%s]: %.2f s", self.step, self.phase, secs)

logger.info("Site=%s image=%s size=%dm", cfg.TEST_SITE_ID, SRTM_IMAGE, cfg.SITE_SIZE_METRES)

## 2. Earth Engine login

Same as notebooks 0.04 and 0.05: the `EE_PROJECT` / service-account variables from the config, otherwise the saved login
from `earthengine authenticate`. If the stored login is missing or invalid, `ee.Authenticate(force=True)` opens a one-time
browser login (not timed). No credential values are printed.

In [ ]:
# changed: credentials come from project config instead of Galileo's GCP_SA_KEY / hard-coded EE_PROJECT
credentials = "persistent"  # saved user login (None would mean no credentials)
if cfg.EE_SERVICE_ACCOUNT_KEY_FILE:
    credentials = ee.ServiceAccountCredentials(cfg.EE_SERVICE_ACCOUNT_EMAIL, cfg.EE_SERVICE_ACCOUNT_KEY_FILE)

def ee_initialise():
    ee.Initialize(credentials=credentials, project=cfg.EE_PROJECT,
                  opt_url="https://earthengine-highvolume.googleapis.com")
    ee.Number(1).getInfo()  # fails here, not later, if the login is invalid

try:
    ee_initialise()
except (ee.EEException, HttpError):  # a stale token surfaces as a 401 HttpError
    if credentials != "persistent":
        raise
    ee.Authenticate(force=True)  # one-time browser login; token is stored outside the project
with timed("Earth Engine initialise", "download"):
    ee_initialise()  # timed separately from any one-off login
print("Earth Engine initialised; login mode:", "persistent user credentials" if credentials == "persistent" else "service account")

## 3. Site geometry

As in notebooks 0.04 and 0.05: the site centre is the Geo-Wiki site's mean location from notebook 0.03's cached summary, and
a box around it is built with Galileo's `EEBoundingBox.from_centre` metres-per-degree formulas. Galileo used 500 m (1 km
tiles); here it is half of `SITE_SIZE_METRES`.

In [ ]:
site = pd.read_csv(cfg.INTERIM_DATA_DIR / "geowiki" / "14871660" / "site_summary.csv",
                   usecols=["sampleid", "longitude", "latitude"]).set_index("sampleid").loc[cfg.TEST_SITE_ID]
lat, lon = float(site["latitude"]), float(site["longitude"])

def metre_per_degree(lat):  # Galileo ee_bbox.py
    m_lat = 111132.954 - 559.822 * cos(radians(2 * lat)) + 1.175 * cos(radians(4 * lat)) - 0.0023 * cos(radians(6 * lat))
    m_lon = 111412.84 * cos(radians(lat)) - 93.5 * cos(radians(3 * lat)) + 0.118 * cos(radians(5 * lat))
    return m_lat, m_lon

m_lat, m_lon = metre_per_degree(lat)
half = cfg.SITE_SIZE_METRES / 2  # changed: Galileo uses EXPORTED_HEIGHT_WIDTH_METRES / 2 = 500
min_lat, max_lat = lat - half / m_lat, lat + half / m_lat
min_lon, max_lon = lon - half / m_lon, lon + half / m_lon
polygon = ee.Geometry.Polygon([[[min_lon, min_lat], [min_lon, max_lat], [max_lon, max_lat], [max_lon, min_lat]]])
print(f"Site {cfg.TEST_SITE_ID}: lat={lat:.5f} lon={lon:.5f}")

## 4. SRTM elevation and slope (lazy, on Earth Engine)

Adapted from Galileo `srtm.py`: the image is clipped to the area, the `elevation` band is selected, and `ee.Terrain.slope`
computes slope in degrees from that elevation. The two bands are stacked and converted to double. Because the slope is
computed after the clip, the slope kernel has no neighbours beyond the area edge; Galileo does the same, and the effect is
checked after download. This only builds the request; nothing is computed until download.

In [ ]:
def get_single_srtm_image(region):  # Galileo srtm.py (the unused start/end date arguments are dropped)
    elevation = ee.Image(SRTM_IMAGE).clip(region).select(SRTM_BANDS[0])
    slope = ee.Terrain.slope(elevation)
    return ee.Image.cat([elevation, slope]).toDouble()

with timed("Build Earth Engine graph", "download"):
    ee_image = get_single_srtm_image(polygon)
print("Bands:", SRTM_BANDS)

## 5. Download

As in notebooks 0.04 and 0.05 and Galileo's `url` mode: `getDownloadURL` requests a GeoTIFF at 10 m scale, so the 30 m
SRTM pixels are resampled to the 10 m grid (nearest neighbour is Earth Engine's default), and the response is streamed to
disk. The request is repeated `N_DOWNLOAD_REPEATS` times to expose latency variability. Bands are labelled `elevation` and
`slope` after the download; that is the only edit to the raw file.

In [ ]:
tif_path = SITE_DIR_RAW / f"{cfg.TEST_SITE_ID}_srtm.tif"
download_secs = []
for i in range(N_DOWNLOAD_REPEATS):
    t0 = time.perf_counter()
    url = ee_image.getDownloadURL({"region": polygon, "scale": 10, "filePerBand": False, "format": "GEO_TIFF"})
    t_url = time.perf_counter()
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with tif_path.open("wb") as f:
        shutil.copyfileobj(r.raw, f)
    t1 = time.perf_counter()
    timings += [{"step": f"Request download URL (run {i+1})", "phase": "download", "seconds": t_url - t0},
                {"step": f"Transfer GeoTIFF (run {i+1})", "phase": "download", "seconds": t1 - t_url}]
    download_secs.append(t1 - t0)
    logger.info("Download run %d: url %.2f s, transfer %.2f s, %d bytes", i + 1, t_url - t0, t1 - t_url, tif_path.stat().st_size)
print("Download seconds per run:", [round(s, 2) for s in download_secs])

# changed: Galileo's GeoTIFFs have no band names; label the bands
with rasterio.open(tif_path, "r+") as src:
    src.descriptions = SRTM_BANDS
    print(src.count, "bands:", src.descriptions)

## 6. Preprocessing

Mirrors the space-band branch of Galileo's `Dataset._tif_to_array`: read the GeoTIFF, reshape `c h w -> h w c`, replace
NaN/Inf with the per-band mean (`_fillna`, 3D branch), normalise (`elevation / 2000`, `slope / 50`), cast to float16 and
save to HDF5 (`save_h5py`, dataset name `sp_x` as in Galileo's space array). `changed:` the Dynamic World/WorldCereal
channels that Galileo stores in the same array are not part of this project, and the fixed shift/div values are used
(Galileo's `std=False` path) since the `normalization.json` mean/std file is not available for this dataset. There is no
time axis and no month array for a static layer.

In [ ]:
def fillna(data, n_bands):  # Galileo dataset.py _fillna (3D branch: h, w, band)
    if data.shape[-1] != n_bands:
        raise ValueError(f"Expected {n_bands} bands - got {data.shape[-1]}")
    is_bad = np.isnan(data) | np.isinf(data)
    if not is_bad.any():
        return data
    data = np.nan_to_num(data, nan=np.nan, posinf=np.nan, neginf=np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_per_band = np.nanmean(data, axis=(0, 1))
    mean_per_band = np.nan_to_num(mean_per_band, nan=0, posinf=0, neginf=0)
    fill = repeat(mean_per_band, "b -> h w b", h=data.shape[0], w=data.shape[1]) * is_bad
    return np.nan_to_num(data, nan=0, posinf=0, neginf=0) + fill

with timed("Read GeoTIFF", "preprocess"):
    with xr.open_dataarray(tif_path, engine="rasterio") as da:  # Galileo uses rioxarray.open_rasterio
        values = da.values  # [bands, H, W]
with timed("Reshape, fill NaN, normalise, float16", "preprocess"):
    sp = rearrange(values, "c h w -> h w c")
    raw_sp = sp.copy()
    n_nan = int(np.isnan(sp).sum())
    sp = fillna(sp, len(SRTM_BANDS))
    sp = ((sp - SRTM_SHIFT_VALUES) / SRTM_DIV_VALUES).astype(np.half)
h5_path = SITE_DIR_PROCESSED / "srtm.h5"
with timed("Save HDF5", "preprocess"):
    with h5py.File(h5_path, "w") as hf:  # Galileo save_h5py (SRTM only; dataset name sp_x)
        hf.create_dataset("sp_x", data=sp)
print(f"Array [h, w, c] = {sp.shape}, dtype {sp.dtype}; NaN/Inf cells before fill: {n_nan} of {raw_sp.size}")
print("Raw value ranges:", {b: (round(float(np.nanmin(raw_sp[..., i])), 1), round(float(np.nanmax(raw_sp[..., i])), 1))
                            for i, b in enumerate(SRTM_BANDS)})

## 7. Edge effect check on slope

Slope is computed after the clip, so pixels near the area edge may be masked or based on fewer neighbours. This cell shows
which raw pixels were missing (before the fill) and how the slope looks along the edge versus the interior.

In [ ]:
slope_raw = raw_sp[..., 1]
edge = np.zeros(slope_raw.shape, bool)
edge[:3, :] = edge[-3:, :] = edge[:, :3] = edge[:, -3:] = True  # 3 pixels of 10 m = one SRTM pixel
print("Missing slope pixels:", int(np.isnan(slope_raw).sum()), "of", slope_raw.size,
      "| missing on the edge ring:", int(np.isnan(slope_raw[edge]).sum()), "of", int(edge.sum()))
print("Mean slope, edge ring vs interior (degrees):", round(float(np.nanmean(slope_raw[edge])), 2),
      round(float(np.nanmean(slope_raw[~edge])), 2))

## 8. Timing and storage summary

Totals split into download (login, graph, URL request, transfer) and preprocessing. Storage is the on-disk size of the raw
GeoTIFF and the processed HDF5. The extrapolation multiplies this single-site, sequential result by a number of sites; it
ignores parallel requests, Earth Engine quota limits and site-to-site variation, so treat it as a rough scale only.

In [ ]:
t = pd.DataFrame(timings)
first_download = t[t.step.str.contains("run 1|initialise|graph")].seconds.sum()
mean_repeat = float(np.mean(download_secs))
pre = t[t.phase == "preprocess"].seconds.sum()
storage = pd.DataFrame({"file": [tif_path.name, h5_path.name], "kind": ["raw GeoTIFF", "processed HDF5"],
                        "bytes": [tif_path.stat().st_size, h5_path.stat().st_size]})
storage["MB"] = storage.bytes / 1e6
summary = pd.DataFrame([
    {"metric": "Download incl. login, graph, first request (s)", "value": first_download},
    {"metric": "Mean download per request (s)", "value": mean_repeat},
    {"metric": "Preprocessing (s)", "value": pre},
    {"metric": "Per-site time, steady state (s)", "value": mean_repeat + pre},
    {"metric": "Raw GeoTIFF (MB)", "value": storage.MB[0]},
    {"metric": "Processed HDF5 (MB)", "value": storage.MB[1]},
])
display(t.round(3)); display(storage[["kind", "bytes", "MB"]].round(4)); display(summary.round(3))
N = 1000
per_site = mean_repeat + pre
print(f"Extrapolation per {N} sites, sequential: {per_site * N / 3600:.2f} h; "
      f"raw {storage.MB[0] * N / 1e3:.2f} GB; processed {storage.MB[1] * N / 1e3:.2f} GB")
t.assign(site_id=cfg.TEST_SITE_ID, modality="srtm").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_srtm_timings.csv", index=False)
storage.assign(site_id=cfg.TEST_SITE_ID, modality="srtm").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_srtm_storage.csv", index=False)
logger.info("Per-site steady-state %.2f s; raw %d B; processed %d B", per_site, storage.bytes[0], storage.bytes[1])
log_ctx.__exit__(None, None, None);

## 9. Conclusions

Result of the run on 2026-10-29 (single site, one machine, sequential; timings vary between requests):

- **Achievable and cheap.** Elevation and slope came back as a 2-band raster (32 x 31 pixels at 10 m, EPSG:4326). Elevation is
  329-337 m and slope 0-5 degrees, so this site is nearly flat.
- **Time.** About 2 s per site for download (request about 0.6-1 s, transfer about 1-1.3 s) plus about 0.13 s preprocessing,
  plus a one-off 1.4 s login, so roughly 0.6 h per 1,000 sites if run one after another: the cheapest modality so far.
- **Storage.** About 2.7 KB raw GeoTIFF and 6 KB processed HDF5 per site (the processed file is larger only because of HDF5
  overhead on a tiny array), i.e. negligible per 1,000 sites.
- **Slope edge effect.** Slope is computed after the clip (as in Galileo), so the outer ring of about one SRTM pixel (3 pixels
  of 10 m) has no slope: 292 of 992 slope pixels (29%) were missing and were mean-filled by the preprocessing. The 100 x 100 m
  Geo-Wiki site in the centre has a 100 m margin and is not affected, but the input margin is. Computing the slope before
  the clip would avoid it, at the cost of departing from Galileo.
- **Grid alignment.** The DEM raster does not share the grid of the Sentinel-1/2 rasters: those two are identical (31 x 31
  pixels), while the DEM is 32 x 31 and offset by a few metres (west edge 3.4e-5 degrees, south edge 4.6e-5 degrees further out).
  Galileo avoids this by exporting all modalities in one image; with separate downloads the DEM must be requested on the
  Sentinel grid (or resampled) before the modalities are stacked. This needs a decision before building the full dataset.
- **Not yet assessed.** Request limits for many sites, parallel downloads, and variation between sites and regions.